<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_37_fastapi_basics/note_lesson_37_fastapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Урок 37 — FastAPI basics: новинний агрегатор, крок 2

| Крок | Що робимо |
|---|---|
| 0 | завантажуємо `news_hub` з API і відкриваємо його `TestClient`-ом |
| 1 | перші запити: `/health`, збір зі знімка, фільтри (вправи 1–2) |
| 2 | `422`: як FastAPI перевіряє параметри (вправа 3) |
| 3 | свої ендпоінти: параметр шляху, тіло запиту (вправи 4–6) |
| 4 | `Depends` і підміна залежності (вправа 7) |
| 5 | OpenAPI (вправа 8) |
| 6 | async чи blocking — вимір (вправа 9) |
| 7 | знайди помилку: чужий URL (вправа 10) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Рефакторинги, архітектура й виміри на сервері — у книзі курсу: [Урок 37](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_37/).

---
## 0. Проєкт у ноутбуці

`news_hub` з уроку 36 + `api.py` (FastAPI), `store.py` (сховище) і `scraper.py` (aiohttp). Сервер запускати не треба: **`TestClient`** надсилає запити прямо в застосунок — ті самі, що пішли б через мережу. `with TestClient(app)` запускає `lifespan`: створюється порожнє сховище.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_37_fastapi_basics"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiohttp, bs4, fastapi, httpx  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())

import asyncio
import time
from typing import Literal

import httpx
from fastapi import Depends, FastAPI, Path, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, HttpUrl, ValidationError, field_validator

from news_hub.api import StoreDep, app, get_scrapers
from news_hub.parser import RawNews
from news_hub.scraper import PageResult, ScrapeOutcome

client = TestClient(app)
client.__enter__()          # те саме, що with TestClient(app) as client: — але на весь ноутбук
print(client.get("/health").json(), "| маршрутів:", len(app.routes))

---
## 1. Перші запити

`client.get(path, params={...})`, `client.post(path, json={...})` — як у `httpx` (урок 31). Відповідь: `.status_code`, `.json()`, `.headers`.

### Вправа 1

Збери новини зі знімка: `POST /api/scrape` з тілом `{"source": "snapshot"}`. Відповідь — у `report`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
report = client.post("/api/scrape", json={"source": "snapshot"}).json()
# END SOLUTION

print({key: report[key] for key in ("news_found", "news_valid", "news_saved", "news_total")})
assert report["news_saved"] == 168 or report["news_total"] == 168
assert client.get("/api/news/count").json() == {"count": 168}
print("✅ Вправа 1 пройдена")

**🔮 Прогноз:** Що покаже `news_saved`, якщо зібрати знімок ще раз?

<details>
<summary>Відповідь</summary>

`0`: ключ сховища — `url`, а всі 168 новин уже є. `news_found` і `news_valid` — знову 168, `news_total` — 168.

</details>

### Вправа 2

Отримай **усі** новини російською (`lang=ru`) — параметр `limit` за замовчуванням 50, тож його доведеться збільшити. Результат (список словників) — у `ru_news`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
ru_news = client.get("/api/news", params={"lang": "ru", "limit": 1000}).json()
# END SOLUTION

assert len(ru_news) == client.get("/api/news/stats").json()["lang"]["ru"] == 138
assert all(news["lang"] == "ru" for news in ru_news)
print(len(ru_news), ru_news[0]["title"])
print("✅ Вправа 2 пройдена")

---
## 2. `422`: помилка перевірки

`limit: int = Query(50, ge=1, le=1000)` — FastAPI перевіряє параметр **до** виклику функції. Помилка — `422` і список `detail`: `loc` (де), `msg` (що), `input` (що прийшло).

### Вправа 3

Надішли `GET /api/news?limit=5000` і збережи відповідь у `response`. Витягни з неї `loc` і `msg` першої помилки.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
response = client.get("/api/news", params={"limit": 5000})
loc = response.json()["detail"][0]["loc"]
msg = response.json()["detail"][0]["msg"]
# END SOLUTION

assert response.status_code == 422
assert loc == ["query", "limit"]
print(loc, msg)
print("✅ Вправа 3 пройдена")

---
## 3. Свої ендпоінти

Невеликий застосунок `mini` — окремо від агрегатора, щоб експериментувати.

### Вправа 4 — параметр шляху

`GET /square/{n}` → `{"n": n, "square": n * n}`. `n` — ціле, не менше 0 (`Path(ge=0)`).

In [ ]:
mini = FastAPI()

# YOUR CODE HERE
# BEGIN SOLUTION
@mini.get("/square/{n}")
async def square(n: int = Path(ge=0)) -> dict[str, int]:
    return {"n": n, "square": n * n}
# END SOLUTION

mini_client = TestClient(mini)
assert mini_client.get("/square/12").json() == {"n": 12, "square": 144}
assert mini_client.get("/square/-1").status_code == 422
assert mini_client.get("/square/abc").status_code == 422
print(mini_client.get("/square/12").json())
print("✅ Вправа 4 пройдена")

**🔮 Прогноз:** Чим параметр шляху (`/square/12`) відрізняється від параметра запиту (`/api/news?limit=12`)? Коли що обирати?

<details>
<summary>Відповідь</summary>

Шлях визначає **ресурс** (яку саме річ: новину, користувача, число) — без нього адреси не існує. Параметр запиту **уточнює** вибірку: фільтр, сортування, сторінка — і зазвичай має значення за замовчуванням (урок 32).

</details>

### Вправа 5 — тіло запиту

`POST /headline` приймає JSON-модель `Headline`: `title` — 10–300 символів, `lang` — `"uk"` або `"ru"`. Повертає `{"words": кількість слів у заголовку, "lang": lang}`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
class Headline(BaseModel):
    title: str = Field(min_length=10, max_length=300)
    lang: Literal["uk", "ru"]


@mini.post("/headline")
async def headline(item: Headline) -> dict[str, int | str]:
    return {"words": len(item.title.split()), "lang": item.lang}
# END SOLUTION

ok = mini_client.post("/headline", json={"title": "Уряд затвердив новий бюджет", "lang": "uk"})
bad = mini_client.post("/headline", json={"title": "Коротко", "lang": "en"})
assert ok.json() == {"words": 4, "lang": "uk"}
assert bad.status_code == 422 and {e["loc"][-1] for e in bad.json()["detail"]} == {"title", "lang"}
print(ok.json(), [e["loc"] for e in bad.json()["detail"]])
print("✅ Вправа 5 пройдена")

### Вправа 6 — ендпоінт агрегатора

Додай в агрегатор `GET /api/news/sources` → словник «джерело → кількість новин». Сховище отримай через `StoreDep`; порахувати допоможе `store.stats()`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
@app.get("/api/news/sources", tags=["news"])
async def news_sources(store: StoreDep) -> dict[str, int]:
    return store.stats()["source"]
# END SOLUTION

sources = client.get("/api/news/sources").json()
assert sources == {"rbc.ua": 167, "auto.rbc.ua": 1}
print(sources)
print("✅ Вправа 6 пройдена")

---
## 4. `Depends` і підміна залежності

`POST /api/scrape` з `source="live"` отримує скрапер через `Depends(get_scrapers)`. У тесті мережа не потрібна: `app.dependency_overrides[get_scrapers] = ...` підставляє заглушку.

### Вправа 7

Напиши заглушку `fake_scraper`, яка «знаходить» дві новини — `good` і `short` (заголовок коротший за 10 символів) — і підстав її для обох режимів. Потім збери `POST /api/scrape` з `{"mode": "sequential"}`.

In [ ]:
good: RawNews = {"title": "Гривня зміцнилася до долара на міжбанку", "url": "https://www.rbc.ua/ukr/news/hryvnia-777.html",
                 "category": "", "description": "", "datetime": "09:05"}
short: RawNews = {"title": "Коротко", "url": "https://www.rbc.ua/ukr/news/short-778.html",
                  "category": "", "description": "", "datetime": ""}

# YOUR CODE HERE
# BEGIN SOLUTION
async def fake_scraper(pages: list[str] | None) -> ScrapeOutcome:
    page = PageResult(url="https://www.rbc.ua/ukr/news/", start=0.0, end=0.1, count=2)
    return ScrapeOutcome(pages=[page], news=[good, short], total_time=0.1)


app.dependency_overrides[get_scrapers] = lambda: {"async": fake_scraper, "sequential": fake_scraper}
live = client.post("/api/scrape", json={"mode": "sequential"}).json()
# END SOLUTION
app.dependency_overrides.clear()

assert (live["news_found"], live["news_valid"], live["news_saved"]) == (2, 1, 1)
assert live["rejected"][0].startswith(short["url"])
print({key: live[key] for key in ("news_found", "news_valid", "news_saved", "news_total")})
print(live["rejected"])
print("✅ Вправа 7 пройдена")

---
## 5. OpenAPI

`GET /openapi.json` — опис API, з якого будуються `/docs` і колекції Postman.

### Вправа 8

Зі схеми OpenAPI збери множину `news_paths` — шляхи, у яких **хоч одна** операція має тег `"news"`.

In [ ]:
schema = client.get("/openapi.json").json()

# YOUR CODE HERE
# BEGIN SOLUTION
news_paths = {path for path, methods in schema["paths"].items()
              if any("news" in operation.get("tags", []) for operation in methods.values())}
# END SOLUTION

assert news_paths == {"/api/news", "/api/news/count", "/api/news/stats", "/api/news/sources"}
print(sorted(news_paths))
print("✅ Вправа 8 пройдена")

---
## 6. Async чи blocking

`httpx.AsyncClient` з `ASGITransport` надсилає запити в застосунок **одночасно** (`asyncio.gather`) — як багато клієнтів до сервера.

In [ ]:
lab = FastAPI()


@lab.get("/blocking")
async def blocking() -> dict[str, str]:
    time.sleep(0.3)                 # займає цикл подій
    return {"kind": "blocking"}


@lab.get("/awaiting")
async def awaiting() -> dict[str, str]:
    await asyncio.sleep(0.3)        # віддає цикл подій іншим
    return {"kind": "awaiting"}


async def measure(path: str, n: int = 5) -> float:
    transport = httpx.ASGITransport(app=lab)
    async with httpx.AsyncClient(transport=transport, base_url="http://lab") as http:
        start = time.perf_counter()
        await asyncio.gather(*(http.get(path) for _ in range(n)))
        return round(time.perf_counter() - start, 2)

**🔮 Прогноз:** 5 одночасних запитів, кожен «чекає» 0.3 с. Скільки триватимуть разом `/blocking` і `/awaiting`?

<details>
<summary>Відповідь</summary>

`/blocking` — ~1.5 с (5 × 0.3: `time.sleep` тримає цикл подій, запити йдуть по черзі). `/awaiting` — ~0.3 с (усі чекають одночасно). На сервері з уроку це 40 с проти 2 с для 20 запитів по 2 с.

</details>

### Вправа 9

Виміряй обидва ендпоінти (`await measure(...)`), результати — у `t_blocking` і `t_awaiting`. Потім додай `/threaded` — **звичайний** `def` з `time.sleep(0.3)` — і виміряй його (`t_threaded`).

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
t_blocking = await measure("/blocking")
t_awaiting = await measure("/awaiting")


@lab.get("/threaded")
def threaded() -> dict[str, str]:
    time.sleep(0.3)                 # def → FastAPI виконає в пулі потоків
    return {"kind": "threaded"}


t_threaded = await measure("/threaded")
# END SOLUTION

print(f"blocking {t_blocking} с | awaiting {t_awaiting} с | threaded {t_threaded} с")
assert t_blocking >= 1.4
assert t_awaiting < 0.8 and t_threaded < 0.8
print("✅ Вправа 9 пройдена")

---
## 7. Знайди помилку

Студент додав ендпоінт «зібрати одну сторінку»:

```python
@app.post("/api/scrape/page")
async def scrape_page(url: str, store: StoreDep) -> dict[str, int]:
    html = requests.get(url, timeout=15).text
    ...
```

Дві проблеми: `requests.get` у `async def` блокує цикл подій (вправа 9), і `url: str` — будь-яка адреса, зокрема внутрішня (`http://localhost:…`): сервер завантажить її на прохання будь-кого (**SSRF**).

### Вправа 10

Опиши модель `PageRequest` з полем `url: HttpUrl` і валідатором: дозволені лише домени, що закінчуються на `rbc.ua`, інакше `ValueError`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
class PageRequest(BaseModel):
    url: HttpUrl

    @field_validator("url")
    @classmethod
    def only_rbc(cls, url: HttpUrl) -> HttpUrl:
        if not (url.host or "").endswith("rbc.ua"):
            raise ValueError(f"лише rbc.ua, а не {url.host}")
        return url
# END SOLUTION

assert PageRequest(url="https://auto.rbc.ua/rus/news/x.html").url.host == "auto.rbc.ua"
for bad in ("http://localhost:8000/api/news", "http://169.254.169.254/latest/meta-data/", "не URL"):
    try:
        PageRequest(url=bad)
    except ValidationError as error:
        print(bad, "→", error.errors()[0]["msg"])
    else:
        raise AssertionError(f"{bad} пропущено")
print("✅ Вправа 10 пройдена")

---
## Самоперевірка

1. Звідки FastAPI бере значення параметра: шлях, рядок запиту, тіло?
2. Що означає `422` і де шукати причину?
3. Навіщо сховище через `Depends`, а не глобальна змінна?
4. Коли ендпоінт писати як `async def`, а коли як `def`?
5. Що таке SSRF і як від нього захищає `ScrapeRequest`?

<details>
<summary>Відповіді</summary>

1. `{n}` у шляху — параметр шляху; простий тип без `{}` — рядок запиту (`?limit=`); Pydantic-модель — JSON-тіло; `Depends` — залежність.
2. Дані не пройшли перевірку; `detail[i].loc` — де (`["query", "limit"]`), `msg` — що не так.
3. Залежність підміняється в тестах (`dependency_overrides`) і замінюється іншою реалізацією (SQLAlchemy в уроці 38) без змін в ендпоінтах.
4. `async def` — коли всередині лише `await` (aiohttp, asyncio.sleep, async-драйвер бази). Є блокуючий виклик — `def` (FastAPI виконає в пулі потоків) або async-бібліотека.
5. Server-Side Request Forgery: сервер завантажує адресу, яку підсунув клієнт, — зокрема внутрішню. `ScrapeRequest` пропускає лише сторінки rbc.ua.

</details>

## Далі

- Книга курсу: [Урок 37](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_37/) — три рефакторинги, архітектура, `/docs`, Postman, виміри `fastapi_demo`.
- У папці `news_hub`: `uvicorn news_hub.api:app --reload` → http://127.0.0.1:8000/docs; `pytest`; колекція `postman/`.
- **Урок 38** — `NewsStore` → SQLAlchemy: новини в базі.

In [ ]:
client.__exit__(None, None, None)   # зупинити lifespan (як вихід з with)